In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/Othercomputers/My PC/Neural_Chess/NeuralChess.Engine/ModelTraining"

!pip install -r requirements.txt

Mounted at /content/drive
/content/drive/Othercomputers/My PC/Neural_Chess/NeuralChess.Engine/ModelTraining
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler, autocast
from model import ChessValueNet
from loss_function import WDL_BCE_Loss
from dataset_manager import ChunkedChessDataset
from datetime import datetime
import shutil
import os

GPU = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 20

drive_dataset_path = "/content/drive/Othercomputers/My PC/Neural_Chess/NeuralChess.Engine/ModelTraining/train_dataset_chunks"
local_dataset_path = "/content/local_train_chunks"

if not os.path.exists(local_dataset_path):
  shutil.copytree(drive_dataset_path, local_dataset_path)
else:
  pass

device = torch.device(GPU)
model = ChessValueNet().to(device)

criterion = WDL_BCE_Loss(scaling_factor=4.0)

optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

scaler = GradScaler(device=device.type)

dataset = ChunkedChessDataset(folder_path=local_dataset_path)
train_loader = DataLoader(
        dataset,
        batch_size=16384,
        pin_memory=True,
        num_workers=2,
        prefetch_factor=4
    )

model.train()
print(f"Starting training loop on device: {device}")

for epoch in range(1, EPOCHS + 1):
  running_loss = 0.0
  batch_count = 0

  for batch_boards, batch_scores in train_loader:
    batch_boards = batch_boards.to(device, non_blocking=True)
    batch_scores = batch_scores.to(device, non_blocking=True).float().view(-1, 1)

    optimizer.zero_grad()

    with autocast(device_type=device.type, dtype=torch.float16):
      predictions = model(batch_boards)
      loss = criterion(predictions, batch_scores)

      scaler.scale(loss).backward()
      scaler.step(optimizer)
      scaler.update()

      running_loss += loss.item()
      batch_count += 1

      if batch_count % 500 == 0:
        print(f"Epoch {epoch} | Batch {batch_count} | Current Loss: {loss.item():.4f} | Time: {datetime.now()}")

  scheduler.step()
  avg_loss = running_loss / batch_count

  log_path = "/content/drive/Othercomputers/My PC/Neural_Chess/NeuralChess.Engine/ModelTraining/batch_results.txt"
  with open(log_path, "a") as myfile:
    myfile.write(f"Epoch {epoch} Complete! Average Loss: {avg_loss:.4f}\n")
  print(f"=== Epoch {epoch} Complete! Average Loss: {avg_loss:.4f} ===")

  save_path = f"/content/drive/Othercomputers/My PC/Neural_Chess/NeuralChess.Engine/ModelTraining/chess_model_epoch_{epoch}.pth"
  torch.save(model.state_dict(), save_path)
  print(f"Weights for Epoch {epoch} successfully backed up to disk!")

save_path = "/content/drive/Othercomputers/My PC/Neural_Chess/NeuralChess.Engine/ModelTraining/chess_model_weights_test.pth"
torch.save(model.state_dict(), save_path)
print("Weights successfully saved to disk!")